In [1]:
import os
import sys
from pathlib import Path
from pyspark.sql import functions as F, types as T

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark
from src.lake import RAW, BRONZE, SILVER, read_delta, write_bronze, with_delta_safe_columns
from src.ingestion_helpers import (
    utc_to_local,
    rename_columns,
    quality_filter,
    write_rejects,
    merge_to_silver,
)

spark = create_spark("incremental-updates")

:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6764f8c1-f2ed-4a31-ba8e-48b6ca3d47d0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 126ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     

## Air Quality — incremental update

In [2]:
aq_update_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "air_quality" / "hourly_88101_update.csv"))
    .filter(F.col("State Code") == 36)
)
aq_update_raw = with_delta_safe_columns(aq_update_raw)

aq_bronze = (
    aq_update_raw
    .withColumn("state_code", F.col("State_Code"))
    .withColumn("measurement_date", F.to_date(F.col("Date_GMT"), "yyyy-MM-dd"))
    .drop("State_Code")
)
write_bronze(aq_bronze, "air_quality", partition_by=["state_code", "measurement_date"])
print(f"[bronze/air_quality] update written")

26/09/24 16:17:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[bronze/air_quality] update written


In [3]:
aq_renamed = rename_columns(
    aq_update_raw,
    {
        "County_Code": "county_code",
        "Site_Num": "site_num",
        "Parameter_Name": "parameter",
        "Date_GMT": "date_gmt",
        "Time_GMT": "time_gmt",
        "Sample_Measurement": "value",
        "Units_of_Measure": "unit",
        "aqi": "aqi",
    },
)

aq_silver_prep = (
    aq_renamed.withColumn(
        "measurement_timestamp",
        utc_to_local(
            F.to_timestamp(
                F.concat_ws(
                    " ",
                    F.date_format(F.col("date_gmt").cast("date"), "yyyy-MM-dd"),
                    F.date_format(F.col("time_gmt").cast("timestamp"), "HH:mm:ss"),
                ),
                "yyyy-MM-dd HH:mm:ss",
            )
        ),
    )
    .withColumn("measurement_date", F.to_date("measurement_timestamp"))
    .withColumn("measurement_hour", F.hour("measurement_timestamp"))
    .select(
        F.col("state_code").cast("int").alias("state_code"),
        F.col("county_code").cast("int").alias("county_code"),
        F.col("site_num").cast("int").alias("site_num"),
        F.col("parameter").cast("string").alias("parameter"),
        F.col("value").cast("double").alias("value"),
        F.col("unit").cast("string").alias("unit"),
        F.col("aqi").cast("int").alias("aqi"),
        "measurement_timestamp",
        "measurement_date",
        "measurement_hour",
    )
)

aq_pk = ["state_code", "county_code", "site_num", "parameter", "measurement_timestamp"]
merge_to_silver(
    spark=spark,
    update_df=aq_silver_prep,
    table_name="air_quality",
    primary_key=aq_pk,
    not_null=["state_code", "county_code", "site_num", "parameter", "measurement_timestamp"],
    numeric_range={"value": {"min": -50, "max": 1000}},
)

[silver/air_quality] Successfully merged update stream. Total rows=113,980 (Total Rejects=4,600)


## Weather — incremental update

In [4]:
weather_update_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "weather" / "weather_update.csv"))
)
weather_update_raw = with_delta_safe_columns(weather_update_raw)

write_bronze(weather_update_raw, "weather")
print(f"[bronze/weather] update written")

[bronze/weather] update written


In [5]:
weather_silver_prep = (
    weather_update_raw
    .withColumn(
        "obs_timestamp",
        F.make_timestamp(
            F.col("year"),
            F.col("month"),
            F.col("day"),
            F.col("hour"),
            F.lit(0),
            F.lit(0),
        ),
    )
    .withColumn("observation_date", F.to_date("obs_timestamp"))
    .withColumn("observation_hour", F.hour("obs_timestamp"))
    .select(
        F.lit("72505394728").alias("station_id"),
        "obs_timestamp",
        F.col("temp").cast("double").alias("temperature_c"),
        (F.col("wspd").cast("double") / F.lit(3.6)).alias("wind_speed_ms"),
        F.col("humidity").cast("double").alias("humidity"),
        "observation_date",
        "observation_hour",
    )
)

weather_pk = ["station_id", "obs_timestamp"]
merge_to_silver(
    spark=spark,
    update_df=weather_silver_prep,
    table_name="weather",
    primary_key=weather_pk,
    not_null=["station_id", "obs_timestamp"],
    partition_by=["observation_date"],
)

[silver/weather] Successfully merged update stream. Total rows=8,872 (Total Rejects=0)
